In [1]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
from scipy.stats import rankdata


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *

In [3]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/esm2_ProGym_matrices.pickle',
          'rb') as f:
    esm_dict = pickle.load(f)

names = list(pg_dict.keys())

In [4]:
pg_dict[names[3]].keys()

dict_keys(['sequence', 'DMS', 'log_probs', 'ref_log_probs', 'llr_matrix', 'spearman_DMS', 'log_probs_forward', 'ref_log_probs_forward', 'llr_matrix_forward', 'log_probs_backward', 'ref_log_probs_backward', 'llr_matrix_backward'])

In [17]:
i = 20
name = names[i]

lp = pg_dict[name]['log_probs']
llr = pg_dict[name]['llr_matrix']
rp = pg_dict[name]['ref_log_probs']
dms = pg_dict[name]['DMS']

lp_for = pg_dict[name]['log_probs_forward']
lp_back = pg_dict[name]['log_probs_backward']

exp_lp_for = np.exp(lp_for)
exp_lp_back = np.exp(lp_back)

p_tm = (exp_lp_for + exp_lp_back)/2

lp_tm = np.log(p_tm)

llr_tm = lp_tm - rp

In [18]:
sp_val, p = spearman_ignore_nan(dms, llr)
sp_val_tm, p = spearman_ignore_nan(dms, llr_tm)

print(sp_val)
print(sp_val_tm)

0.2726655554389908
0.26234541680416723


In [23]:
count = 0

for i in range(len(names)):  

    name = names[i]
    seq = pg_dict[name]['sequence']
    
    if len(seq) < 1024:

        lp = pg_dict[name]['log_probs']
        llr = pg_dict[name]['llr_matrix']
        rp = pg_dict[name]['ref_log_probs']
        dms = pg_dict[name]['DMS']

        lp_for = pg_dict[name]['log_probs_forward']
        lp_back = pg_dict[name]['log_probs_backward']

        exp_lp_for = np.exp(lp_for)
        exp_lp_back = np.exp(lp_back)

        p_tm = (exp_lp_for + exp_lp_back)/2

        lp_tm = np.log(p_tm)

        llr_tm = lp_tm - rp

        sp_val, p = spearman_ignore_nan(dms, llr)
        sp_val_tm, p = spearman_ignore_nan(dms, llr_tm)

        if sp_val_tm > sp_val:
            count = count + 1
    
print(count)

        # print(name)
        # print(f"Original spearman: {sp_val}")
        # print(f"True mean spearman: {sp_val_tm}")
        # print()

10


In [13]:
print(lp_tm)

print(lp)

[[-3.245388   -4.4592867  -4.2710595  ... -3.8275363  -6.4039464
  -5.3601747 ]
 [-3.0489798  -4.7897954  -3.7950854  ... -3.5061188  -0.68379486
  -4.670639  ]
 [-3.0382497  -4.7533674  -3.707944   ... -3.7820122  -4.9351315
  -4.3599043 ]
 ...
 [-3.2871907  -4.5647616  -3.672557   ... -3.4132648  -4.8445225
  -3.7227418 ]
 [-3.198079   -4.4690156  -3.4831142  ... -3.3668334  -4.6295514
  -2.72137   ]
 [-3.437845   -4.742127   -3.8323703  ... -2.6876614  -4.9992456
  -4.4426966 ]]
[[-2.671175  -3.9057484 -3.5685139 ... -3.150713  -5.4753895 -4.549303 ]
 [-4.5947556 -4.3755174 -4.8122087 ... -4.777762  -0.3122851 -4.2508225]
 [-3.5998516 -5.1621456 -4.6252117 ... -5.0727215 -6.3548107 -5.3227825]
 ...
 [-3.0720196 -4.4227276 -4.077772  ... -3.3954372 -5.2065716 -3.9509716]
 [-3.0436532 -4.09062   -3.3755624 ... -3.2689798 -4.254118  -2.5459573]
 [-3.4487703 -4.5051513 -4.9875655 ... -1.8363969 -5.3438125 -4.2179503]]


In [ ]:
# with open("/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_conditional10_matrices.pickle", "wb") as f:
#     pickle.dump(pg_dict_cond_10, f)